In [1]:
!pip install evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 3.7 MB/s eta 0:00:00


In [2]:
!pip install -U datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 19.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.2
    Uninstalling fsspec-2025.3.2:
      Successfully uninstalled fsspec-2025.3.2
  Attempting uninstall: datasets
    Found existing installation: datasets 2.14.4
    Uninstalling datasets-2.14.4:
      Successfully uninstalled datasets-2.14.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; pl

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
from collections import Counter
import random
import re
import datasets
import tqdm
import math
from functools import partial
import math
import argparse
import os
import collections
import json
import sentencepiece
import shutil
import copy
import multiprocessing
import transformers
from dataclasses import dataclass, field
from evaluate import load

# set "high" if you have a GPU with compute capability >= 8.0 else "highest"
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

# Training config

In [4]:
## you can modify some options such as batch_size, depending on your environments

training_config = {
    "batch_size": 8,
    "epochs": 3,
    "lr": 1e-3,
    "warmup_steps": 50,
    "device": "cuda" if torch.cuda.is_available() else "cpu",

    "gradient_accumulate_steps": 1,
}

# Dataset load

In [ ]:
#Previous dataset

# dataset = datasets.load_dataset("lemon-mint/korean_english_parallel_wiki_augmented_v1",split="train")
# dataset = dataset.filter(lambda x: len(x['english']) < 8192 and len(x['english']) > 128 and len(x['korean']) < 8192 and len(x['korean']) > 128)
# num_samples = int(len(dataset) * 0.2)
# dataset = dataset.shuffle(seed=42).select(range(num_samples))
# valid_set = dataset.select(range(2000))
# train_set = dataset.select(range(2000, num_samples))

# New dataset for English-German translation
train_set = datasets.load_dataset("bentrevett/multi30k", split="train") #Dataset change to English-German translation
valid_set = datasets.load_dataset("bentrevett/multi30k", split="validation")

tokenizer = transformers.AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-de-en") # Chnaged tokenizer to English-German translation
additional_special_tokens = {}
if tokenizer.pad_token is None:
    additional_special_tokens["pad_token"] = "<pad>"
if tokenizer.eos_token is None:
    additional_special_tokens["eos_token"] = "</s>"
if tokenizer.bos_token is None:
    additional_special_tokens["bos_token"] = "<s>"
tokenizer.add_special_tokens(additional_special_tokens)

def collate_fn(batch):
    english_corpus = [item["en"] for item in batch]
    german_corpus = [item["de"] for item in batch]
    english_corpus = tokenizer(english_corpus, padding=True, truncation=True, return_tensors="pt", max_length=128, pad_to_multiple_of=64) # Here make some changes to max_length beacause the length of the sentences in English-German translation are much shorter
    german_corpus = tokenizer(german_corpus, padding=True, truncation=True, return_tensors="pt", max_length=128, pad_to_multiple_of=64)
    labels = german_corpus["input_ids"].clone()
    labels[german_corpus['attention_mask'].eq(0)] = -100


    return {
        "encoder_input_ids": english_corpus["input_ids"],
        "encoder_attention_mask": english_corpus["attention_mask"],
        "decoder_input_ids": german_corpus["input_ids"],
        "labels": labels,
    }



/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/1.15k [00:00<?, ?B/s]

train.jsonl:   0%|          | 0.00/4.60M [00:00<?, ?B/s]

val.jsonl:   0%|          | 0.00/164k [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/156k [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/1.38k [00:00<?, ?B/s]

source.spm:   0%|          | 0.00/797k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/768k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.27M [00:00<?, ?B/s]

/usr/local/lib/python3.11/dist-packages/transformers/models/marian/tokenization_marian.py:177: UserWarning: Recommended: pip install sacremoses.
  warnings.warn("Recommended: pip install sacremoses.")


### Model implement

In [ ]:
@dataclass
class ModelConfig(object):
    vocab_size: int = field(default=50000)
    encoder_hidden_dim: int = field(default=512) # hidden dimention of encoder lstm
    decoder_hidden_dim: int = field(default=512) # hidden dimention of decoder lstm
    hidden_dim: int = field(default=512) # hidden dimention of other module like attention
    embed_dim: int = field(default=512) # embedding dimention
    pad_idx: int = field(default=0)
    sos_idx: int = field(default=1)
    eos_idx: int = field(default=2)
    n_layers: int = field(default=1)
    dropout: float = field(default=0.1)

    attention_type:str = field(default="global")
    window_size: int = field(default=4) # I changed this to 4, because the sentences in the new dataset are much shorter than the previous one
    sigma_ratio: float = field(default=2.0)

    do_input_feeding: bool = field(default=True)

class GlobalAttention(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.query_proj = nn.Linear(config.decoder_hidden_dim, config.hidden_dim, bias=False)
        self.key_proj = nn.Linear(config.encoder_hidden_dim * 2, config.hidden_dim, bias=False)
        self.value_proj = nn.Linear(config.encoder_hidden_dim * 2, config.hidden_dim, bias=False)
        self.output_proj = nn.Linear(config.hidden_dim, config.decoder_hidden_dim, bias=False)

        self.dropout = nn.Dropout(config.dropout)
        self.scale = float(config.hidden_dim ** 0.5)

    def forward(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        query = self.query_proj(decoder_hidden_query) #(Batch, 1, hidden_dim)
        key = self.key_proj(encoder_outputs) #(Batch, seq_len, hidden_dim)
        value = self.value_proj(encoder_outputs) #(Batch, seq_len, hidden_dim)

        # fill here for global attention forward
        # shape hint:
        # context: (batch, 1, hidden_dim)
        ######

        ## YOUR CODES
        attn_scores = torch.matmul(query, key.transpose(1, 2)) / self.scale

        if encoder_attention_mask is not None:
            mask = encoder_attention_mask.unsqueeze(1)
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        a = torch.softmax(attn_scores, dim=-1)
        a = self.dropout(a)
        context = torch.matmul(a, value)

        output_context = self.output_proj(context)
        return output_context

class LocalAttention(GlobalAttention):
    def __init__(self, config: ModelConfig):
        super().__init__(config)
        self.window_size = config.window_size
        self.location_proj_up = nn.Linear(config.decoder_hidden_dim, config.hidden_dim, bias=False)
        self.location_proj_down = nn.Linear(config.hidden_dim, 1, bias=False)
        self.sigma = float(self.window_size / config.sigma_ratio)

    def forward(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        key, value, attn_mask, gaussian_penalty = self._gather_local_context(decoder_hidden_query, encoder_outputs, encoder_attention_mask)
        query = self.query_proj(decoder_hidden_query)
        key = self.key_proj(key)
        value = self.value_proj(value)

        # fill here for local attention forward
        # shape hint:
        # context: (batch, 1, hidden_dim)
        ######

        ## YOUR CODES
        attn_scores = torch.matmul(query, key.transpose(-2,-1)) / self.scale
        attn_mask = attn_mask.unsqueeze(1)
        attn_scaled = attn_scores.masked_fill(attn_mask == 0, -1e9)

        attn_scores = attn_scaled + torch.log(gaussian_penalty.unsqueeze(1) + 1e-8)
        a = torch.softmax(attn_scores, dim=-1)

        context = torch.matmul(a, value)

        ######
        output_context = self.output_proj(context)

        return output_context

    def _gather_local_context(self, decoder_hidden_query, encoder_outputs, encoder_attention_mask):
        device = encoder_outputs.device
        model_dtype = encoder_outputs.dtype
        src_len = encoder_attention_mask.sum(dim=-1).to(dtype = model_dtype).unsqueeze(-1)
        batch_size, seq_len, hidden_dim = encoder_outputs.size()
        # fill here for local context window
        # shape hint:
        # local_key: (batch, window_size * 2 + 1, hidden_dim)
        # local_value: (batch, window_size * 2 + 1, hidden_dim)
        # local_attn_mask: (batch, window_size * 2 + 1)
        # gaussian_penalty: (batch, window_size * 2 + 1)
        ######
        ## YOUR CODES
        p_t = self.location_proj_up(decoder_hidden_query)
        p_t = self.location_proj_down(torch.tanh(p_t))
        p_t = torch.sigmoid(p_t)
        src_len = src_len.unsqueeze(2)
        p_t = (src_len - 1) * p_t
        p_t = p_t.squeeze(1)  # (batch_size, 1)
        src_len = src_len.squeeze(1)  # (batch_size,1)


        window_range = torch.arange(-self.window_size, self.window_size+1, device=device, dtype=model_dtype)  # (window_size * 2 + 1,)
        window_positions = (p_t + window_range.unsqueeze(0)).long()
        additional_mask = torch.ones((batch_size, self.window_size * 2 + 1), device=device, dtype=model_dtype)
        zero_tensor = torch.tensor(0.0, device=additional_mask.device, dtype=model_dtype)
        additional_mask = torch.where(window_positions < 0, zero_tensor, additional_mask)
        additional_mask = torch.where(window_positions >= src_len.long(), zero_tensor, additional_mask)

        window_positions = torch.clamp(window_positions,
                                  min=torch.tensor(0, device=device),
                                  max=src_len - 1).long()

        batch_indices = torch.arange(batch_size, device=device).unsqueeze(1)
        local_key = encoder_outputs[batch_indices, window_positions]
        local_value = local_key.clone()
        local_attn_mask = encoder_attention_mask[batch_indices, window_positions]
        local_attn_mask = local_attn_mask.float() * additional_mask

        distances = window_range.unsqueeze(0)

        gaussian_penalty = torch.exp(-1 * (distances ** 2) / (2 * self.sigma ** 2))
        gaussian_penalty = gaussian_penalty * additional_mask

        ######

        return local_key, local_value, local_attn_mask, gaussian_penalty

class Encoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.encoder = nn.LSTM(
            input_size=config.embed_dim,
            hidden_size=config.encoder_hidden_dim,
            num_layers=config.n_layers,
            dropout=config.dropout if config.n_layers > 1 else 0,
            bidirectional=True,
            batch_first=True
        )

        self.h_dec_proj = nn.Linear(config.encoder_hidden_dim * 2, config.decoder_hidden_dim)
        self.c_dec_proj = nn.Linear(config.encoder_hidden_dim * 2, config.decoder_hidden_dim)

    def forward(self, input_embeds, attention_mask):

        # Fill here for encoder forward
        # shape hint
        # input_embeds: (batch, src_seq_len, embed_dim)
        # attention_mask: (batch, src_seq_len)
        # encoder_output: (batch, src_seq_len, hidden_dim)
        # h_enc: (n_layers, batch, decoder_hidden_dim)
        # c_enc: (n_layers, batch, decoder_hidden_dim)
        # hint for implementation
        # 1. use nn.utils.rnn.pack_padded_sequence to packing inputs for rnn series, see https://pytorch.org/docs/stable/generated/torch.nn.utils.rnn.pack_padded_sequence.html
        #    failure to properly handle padding will result in a penalty.
        # 2. lstm cell state and hidden state will be doubled because of bidirectional lstm.
        #    decoder will be unidirectional for causal language modeling.
        #    handle the hidden state and cell state to be same as decoder.
        ######

        ## YOUR CODES
        length = attention_mask.sum(dim=1)
        packed_embed = nn.utils.rnn.pack_padded_sequence(input_embeds, length.cpu(), batch_first=True, enforce_sorted=False) #Here we delete the padding and compact our data
        packed_output, (h, c) = self.encoder(packed_embed)
        #h and c size of (2 * n_layers, batch, hidden_size)

        h = h.view(self.config.n_layers, 2, input_embeds.size(0), self.config.encoder_hidden_dim)
        h = torch.cat((h[:, 0], h[:, 1]), dim=-1) # Here we concatenate the forward and backward hidden states so the sizes become (n_layers, batch, hidden_size * 2)

        c = c.view(self.config.n_layers, 2, input_embeds.size(0), self.config.encoder_hidden_dim)
        c = torch.cat((c[:, 0], c[:, 1]), dim=-1) # Here we do the same for cell states

        h_enc = self.h_dec_proj(h)
        c_enc = self.c_dec_proj(c)

        encoder_output, _ = nn.utils.rnn.pad_packed_sequence(packed_output, batch_first=True, total_length=input_embeds.size(1)) # Here we bring all the data back to the original size
        ######

        return encoder_output, (h_enc, c_enc)

class Decoder(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.decoder = nn.LSTM(
            input_size=config.embed_dim + config.hidden_dim if config.do_input_feeding else config.embed_dim,
            hidden_size=config.decoder_hidden_dim,
            num_layers=config.n_layers,
            dropout=config.dropout if config.n_layers > 1 else 0,
            batch_first=True
        )
        match config.attention_type:
            case "local":
                self.attention = LocalAttention(config)
            case "global":
                self.attention = GlobalAttention(config)
            case _:
                raise ValueError(f"Unknown attention type: {config.attention_type}")
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, input_embeds, encoder_outputs, h_enc, c_enc, attention_mask):
        decoder_output, (h_dec, c_dec) = self.decoder(input_embeds, (h_enc, c_enc))
        attention_context = self.attention(decoder_output, encoder_outputs, attention_mask)
        decoder_output = decoder_output + attention_context

        return decoder_output, attention_context, (h_dec, c_dec)

class Seq2Seq(nn.Module):
    def __init__(self, config: ModelConfig):
        super().__init__()
        self.config = config

        self.embedding = nn.Embedding(config.vocab_size, config.embed_dim, padding_idx=config.pad_idx)

        self.encoder = Encoder(config)
        self.decoder = Decoder(config)

        self.lm_head = nn.Linear(config.hidden_dim, config.vocab_size)
        self.dropout = nn.Dropout(config.dropout)

    def forward(self, encoder_input_ids, encoder_attention_mask, decoder_input_ids, labels=None, cache=None):
        if cache is None:
            encoder_input_embeds = self.embedding(encoder_input_ids)
            encoder_outputs, (h_enc, c_enc) = self.encoder(encoder_input_embeds, encoder_attention_mask)

            current_h_dec, current_c_dec = h_enc, c_enc
            prev_attn_context = None
        else:
            encoder_outputs, current_h_dec, current_c_dec, prev_attn_context = cache

        batch_size, tgt_len = decoder_input_ids.shape
        decoder_input_embeds = self.embedding(decoder_input_ids)

        if prev_attn_context is None:
            prev_attn_context = torch.zeros((batch_size, 1, self.config.decoder_hidden_dim)).to(decoder_input_embeds)

        outputs = []

        for t in range(tgt_len):
            # fill here for decoder forward
            ######

            ## YOUR CODES
            current_embed = decoder_input_embeds[:, t:t+1, :]

            if self.config.do_input_feeding:
                decoder_input = torch.cat([current_embed, prev_attn_context], dim=-1)
            else:
                decoder_input = current_embed

            decoder_output, prev_attn_context, (current_h_dec, current_c_dec) = self.decoder(
                decoder_input,
                encoder_outputs,
                current_h_dec,
                current_c_dec,
                encoder_attention_mask
            )
            ######
            outputs.append(decoder_output)


        outputs = torch.cat(outputs, dim=1)

        lm_logits = self.lm_head(outputs)

        loss = None
        if labels is not None:
            # for cross entropy loss
            # loss must be scalar

            labels_for_loss = labels[:, 1:].contiguous()
            lm_logits_for_loss = lm_logits[:, :-1, :].contiguous()
            loss = F.cross_entropy(lm_logits_for_loss.view(-1, self.config.vocab_size), labels_for_loss.view(-1))

            return loss
        else:
            return lm_logits, (encoder_outputs, current_h_dec, current_c_dec, prev_attn_context)

    @torch.no_grad()
    def generate(
        self,
        encoder_input_ids: torch.LongTensor,
        encoder_attention_mask: torch.LongTensor,
        max_new_tokens: int = 128, #I changed this for 128, becuase in the new dataset the sentences are much shorter than the previous one
    ):
        batch_size, _ = encoder_input_ids.shape
        device = encoder_input_ids.device
        eos = self.config.eos_idx

        unfinish_flag = torch.ones(batch_size, dtype=torch.long, device=device)
        cache = None
        decoder_input_ids = torch.full((batch_size, 1), self.config.sos_idx, dtype=torch.long, device=device)

        for _ in range(max_new_tokens):
            # fill here for causal generation
           ######

            ## YOUR CODES
            logits, (encoder_outputs, current_h_dec, current_c_dec, prev_attn_context) = self.forward(
                    encoder_input_ids, encoder_attention_mask, decoder_input_ids[:,-1:], labels=None, cache=cache
                )

            cache = (encoder_outputs, current_h_dec, current_c_dec, prev_attn_context)

            next_token = logits[:, -1, :].argmax(dim=-1)  # (batch_size,)

            next_token = next_token * unfinish_flag + eos * (1 - unfinish_flag)
            next_token = next_token.unsqueeze(1)  # (batch_size, 1)

            decoder_input_ids = torch.cat([decoder_input_ids, next_token], dim=1)

            unfinish_flag = unfinish_flag * (next_token.squeeze(1) != eos).long()

            if unfinish_flag.sum() == 0:
                break
            ######
        return decoder_input_ids


In [7]:
def train(model, train_dataset, valid_dataset, collate_fn, train_args, prefix):
    optimizer = optim.Adam(model.parameters(), lr=train_args["lr"])

    train_dataloader = DataLoader(train_dataset, batch_size=train_args['batch_size'], shuffle=True, collate_fn=collate_fn, num_workers=os.cpu_count())
    valid_dataloader = DataLoader(valid_dataset, batch_size=train_args['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=os.cpu_count())

    total_steps = len(train_dataloader) * train_args['epochs']

    num_training_steps = train_args['epochs'] * (len(train_dataloader) // train_args['gradient_accumulate_steps'])
    scheduler = transformers.get_scheduler(
        name="cosine",
        optimizer=optimizer,
        num_warmup_steps=train_args['warmup_steps'],
        num_training_steps=num_training_steps
    )

    best_loss = 987654321
    optimizer.zero_grad()

    output_path = os.path.join("output", prefix)
    os.makedirs(output_path, exist_ok=True)
    with open(os.path.join(output_path, "train_args.json"), "w") as f:
        json.dump(train_args, f)

    pbar = tqdm.tqdm(total=total_steps, desc="training")
    for epoch in range(train_args['epochs']):
        pbar.set_description(f"Epoch {epoch+1}/{train_args['epochs']}")
        move_avg_loss = []
        model.train()
        for i, batch in enumerate(train_dataloader):
            batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}

            loss = model(**batch)
            loss = loss / train_args['gradient_accumulate_steps']
            if loss.size() != torch.Size([]):
                loss = loss.mean()
            loss.backward()

            if (i+1) % train_args['gradient_accumulate_steps'] == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
                optimizer.zero_grad()
                scheduler.step()

            move_avg_loss.append(loss.item())
            if len(move_avg_loss) > 100: move_avg_loss.pop(0)
            pbar.set_postfix_str(f"loss: {sum(move_avg_loss)/len(move_avg_loss):.04f} lr: {optimizer.param_groups[0]['lr']:.2e}")
            pbar.update(1)

        model.eval()
        with torch.no_grad():
            eval_loss = 0
            for i, batch in enumerate(valid_dataloader):
                batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
                loss_val = model(**batch)
                if loss_val.size() != torch.Size([]):
                    loss_val = loss_val.mean()
                eval_loss += loss_val.item()
                pbar.set_postfix_str(f"val_loss: {eval_loss / (i+1):.04f}")
        eval_loss /= len(valid_dataloader)
        pbar.write(f"Validation Loss: {eval_loss:.04f}")

        if eval_loss < best_loss:
            best_loss = eval_loss

            torch.save(model.state_dict(), os.path.join(output_path,"best_model.pth"))
            pbar.write(f"Model Saved best loss: {best_loss:.04f}")

    pbar.close()

def evaluate(model, dataset, tokenizer, collate_fn, train_args):
    model.eval()
    dataloader = DataLoader(dataset, batch_size=train_args['batch_size'], shuffle=False, collate_fn=collate_fn, num_workers=os.cpu_count())

    answers = []
    predicts = []
    for i, batch in enumerate(tqdm.tqdm(dataloader, desc="Evaluating")):
        batch = {k:v.to(train_args['device']) if isinstance(v,torch.Tensor) else v for k,v in batch.items()}
        gen_output = model.generate(
            encoder_input_ids=batch["encoder_input_ids"],
            encoder_attention_mask=batch["encoder_attention_mask"],
            max_new_tokens=512
        )
        pred = tokenizer.batch_decode(gen_output, skip_special_tokens=True)
        ans = tokenizer.batch_decode(batch["labels"], skip_special_tokens=True)
        answers.extend(ans)
        predicts.extend(pred)

    bleu = load("bleu")
    result = bleu.compute(predictions=predicts, references=answers)
    print(f"BLEU: {result['bleu']:.4f}")

In [ ]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="global",
    do_input_feeding=False,
)

model = Seq2Seq(config).to(training_config["device"])
model = model.to(torch.bfloat16)
model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_global_attention_no_input_feeding"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_global_attention_no_input_feeding", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(58102, 512, padding_idx=58100)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): GlobalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (lm_head): Linear(in_features=512, out_features=58102, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


Epoch 1/3:  33%|███▎      | 3625/10875 [14:40<29:51,  4.05it/s, val_loss: 2.4022]

Validation Loss: 2.4022


Epoch 2/3:  33%|███▎      | 3625/10875 [14:41<29:51,  4.05it/s, val_loss: 2.4022]

Model Saved best loss: 2.4022


...

Validation Loss: 2.0757


Epoch 3/3: 100%|██████████| 10875/10875 [43:48<00:00,  4.14it/s, val_loss: 2.0757]


Model Saved best loss: 2.0757


Evaluating: 100%|██████████| 127/127 [00:07<00:00, 16.10it/s]


BLEU: 0.2263


In [ ]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="global",
)

model = Seq2Seq(config).to(training_config["device"])
model = model.to(torch.bfloat16)
model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_global_attention"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_global_attention", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(58102, 512, padding_idx=58100)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(1024, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): GlobalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (lm_head): Linear(in_features=512, out_features=58102, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)


Epoch 1/3:  33%|███▎      | 3625/10875 [14:44<29:32,  4.09it/s, val_loss: 2.4929]

Validation Loss: 2.4929


Epoch 2/3:  33%|███▎      | 3625/10875 [14:44<29:32,  4.09it/s, val_loss: 2.4929]

Model Saved best loss: 2.4929


...

Validation Loss: 2.1308


Epoch 3/3: 100%|██████████| 10875/10875 [44:14<00:00,  4.10it/s, val_loss: 2.1308]


Model Saved best loss: 2.1308


Evaluating: 100%|██████████| 127/127 [00:07<00:00, 17.57it/s]


BLEU: 0.2158


In [8]:
config = ModelConfig(
    vocab_size=len(tokenizer),
    pad_idx=tokenizer.pad_token_id,
    sos_idx=tokenizer.bos_token_id,
    eos_idx=tokenizer.eos_token_id,
    n_layers=2,
    dropout=0.1,

    attention_type="local",
)

model = Seq2Seq(config).to(training_config["device"])
model = model.to(torch.bfloat16)
model.compile()
print(model)

train(
    model,
    train_set,
    valid_set,
    collate_fn,
    training_config,
    prefix="seq2seq_local_attention"
)

model.load_state_dict(torch.load(os.path.join("output", "seq2seq_local_attention", "best_model.pth")))
evaluate(
    model,
    valid_set,
    tokenizer,
    collate_fn,
    training_config
)

del model
torch.cuda.empty_cache()

Seq2Seq(
  (embedding): Embedding(58102, 512, padding_idx=58100)
  (encoder): Encoder(
    (encoder): LSTM(512, 512, num_layers=2, batch_first=True, dropout=0.1, bidirectional=True)
    (h_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
    (c_dec_proj): Linear(in_features=1024, out_features=512, bias=True)
  )
  (decoder): Decoder(
    (decoder): LSTM(1024, 512, num_layers=2, batch_first=True, dropout=0.1)
    (attention): LocalAttention(
      (query_proj): Linear(in_features=512, out_features=512, bias=False)
      (key_proj): Linear(in_features=1024, out_features=512, bias=False)
      (value_proj): Linear(in_features=1024, out_features=512, bias=False)
      (output_proj): Linear(in_features=512, out_features=512, bias=False)
      (dropout): Dropout(p=0.1, inplace=False)
      (location_proj_up): Linear(in_features=512, out_features=512, bias=False)
      (location_proj_down): Linear(in_features=512, out_features=1, bias=False)
    )
    (dropout): Dropout(p=0.1,

Epoch 1/3:  33%|███▎      | 3625/10875 [16:10<31:28,  3.84it/s, val_loss: 2.9025]

Validation Loss: 2.9025


Epoch 2/3:  33%|███▎      | 3625/10875 [16:10<31:28,  3.84it/s, val_loss: 2.9025]

Model Saved best loss: 2.9025


...

Validation Loss: 2.5320


Epoch 3/3: 100%|██████████| 10875/10875 [47:38<00:00,  3.80it/s, val_loss: 2.5320]


Model Saved best loss: 2.5320


Evaluating: 100%|██████████| 127/127 [00:10<00:00, 11.58it/s]


BLEU: 0.1477
